<a href="https://colab.research.google.com/github/Prktriesitout/LangGraph/blob/main/LanggraphProject2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Business Scenario:

A sales chatbot gathers customer preferences across multiple interactions. It should resume the session even after shutdown.

Problem Statement:

Implement persistence using MemorySaver or SQLiteSaver or any other API of your choice so that agent state (username and interests) is reloaded on restart.

Steps to Follow:

Create a graph with nodes: get_name, get_interest, recommend.
Integrate a persistence layer (MemorySaver or SQLiteSaver).
Run once, stop after 2 nodes, then rerun and resume.
Verify the state is restored correctly.
Expected Output:

After resuming:

Restored State: {'name': 'Ravi', 'interest': 'fitness'}

Final Output: 'Recommended plan: Home Gym Starter Kit.'

In [21]:
! pip install -q langgraph langgraph-checkpoint-sqlite

In [22]:
import sqlite3
from typing import TypedDict
from langgraph.graph import StateGraph , START, END
from langgraph.checkpoint.sqlite import SqliteSaver

In [23]:
DB_PATH = "/content/sales_bot.db"

In [24]:
THREAD = {"configurable": {"thread_id": "customer-ravi"}}
class State(TypedDict):
    name: str
    interest: str
    recommendation: str

def get_name(state: State):
  print("[node] get_name")
  return {"name": "Ravi"}

def get_interest(state: State):
  print("[node] get_interest")
  return {"interest": "fitness"}

def recommend(state: State):
  print("[node] recommend")
  plans = {"fitness": "Home Gym Starter Kit"}
  return {"recommendation": f"Recommended plan: {plans.get(state['interest'], 'Gift Card')}."}

builder = StateGraph(State)
builder.add_node("get_name", get_name)
builder.add_node("get_interest", get_interest)
builder.add_node("recommend",recommend)
builder.add_edge(START,"get_name")
builder.add_edge("get_name","get_interest")
builder.add_edge("get_interest","recommend")
builder.add_edge("recommend" , END)

conn = sqlite3.connect(DB_PATH, check_same_thread=False)
graph = builder.compile(checkpointer=SqliteSaver(conn),
                        interrupt_before=["recommend"])

In [25]:
graph.invoke({"name": "", "interest": "", "recommendation": ""}, THREAD)
print("Saved state:", graph.get_state(THREAD).values)
print("Next node:", graph.get_state(THREAD).next)

[node] get_name
[node] get_interest
Saved state: {'name': 'Ravi', 'interest': 'fitness', 'recommendation': ''}
Next node: ('recommend',)


In [26]:
snap = graph.get_state(THREAD)
print("Restored State:", {"name": snap.values["name"], "interest": snap.values["interest"]})

result = graph.invoke(None, THREAD)
print("invoke returned:", result)   # debug: shows what keys came back

final = graph.get_state(THREAD).values        # always the saved, complete state
print("Final Output:", repr(final.get("recommendation")))

Restored State: {'name': 'Ravi', 'interest': 'fitness'}
[node] recommend
invoke returned: {'name': 'Ravi', 'interest': 'fitness', 'recommendation': 'Recommended plan: Home Gym Starter Kit.'}
Final Output: 'Recommended plan: Home Gym Starter Kit.'
